## Load Libraries

In [ ]:
!pip install nflreadpy -q

import nflreadpy as nfl
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline
from scipy.stats import randint
from sklearn.metrics import mean_absolute_error, r2_score
from functools import reduce
from joblib import dump, load
import operator
import time
import os
from google.colab import drive

## Data Loading

In [ ]:
def loader(load_fn, *args):

    return load_fn(*args).to_pandas()

def nextgen_loader(load_fn):

    dfs = []
    for ngt in ['rushing', 'receiving', 'passing']:
        loaded_df = loader(
            load_fn,
            range(2022, 2026),
            ngt
        ).rename(
            columns={
                'player_gsis_id': 'player_id',
                'player_position': 'position'
            }
        )
        loaded_df = season_update(loaded_df)
        loaded_df = position_update(loaded_df)
        loaded_df = season_week_update(loaded_df)
        dfs.append(loaded_df)

    return pd.concat(dfs, ignore_index=True)

In [ ]:
def season_week_update(df):

    df[['season', 'week']] = df[['season', 'week']].astype('int64')

    return df

def season_update(df):

    return df[df['season_type'] == 'REG']

def position_update(df):

    return df[df['position'].isin(['QB', 'WR', 'TE', 'RB'])]

def add_half_ppr(df):

    df['half'] = df['std'] + (df['receptions'] / 2)

    return df

In [ ]:
player_stats = loader(
    nfl.load_player_stats,
    range(2022, 2026)
).rename(
    columns={
        'fantasy_points': 'std',
        'fantasy_points_ppr': 'full',
        'player_display_name': 'player',
    }
)
player_stats = season_update(player_stats)
player_stats = position_update(player_stats)
player_stats = season_week_update(player_stats)
player_stats = add_half_ppr(player_stats)
team_stats = loader(
    nfl.load_team_stats,
    range(2022, 2026)
)
team_stats = season_update(team_stats)
team_stats = season_week_update(team_stats)
player_info = loader(nfl.load_players)
player_info = player_info.rename(columns={'gsis_id': 'player_id'})
player_info = position_update(player_info)
nextgen_stats = nextgen_loader(nfl.load_nextgen_stats)

## Feature Engineering

In [ ]:
RB_STANDARD_FEATURES = [
    'rushing_yards',
    'rushing_tds',
    'rushing_fumbles_lost',
    'rushing_epa',
    'receptions',
    'receiving_yards',
    'receiving_yards_after_catch',
    'receiving_tds',
    'receiving_fumbles_lost',
    'receiving_epa',
    'wopr',
]
RB_NEXTGEN_FEATURES = [
    'avg_rush_yards',
    'expected_rush_yards',
    'rush_yards_over_expected',
]
RB_DEFENSE_FEATURES = [
    'rushing_yards',
    'rushing_tds',
    'rushing_fumbles_lost',
    'rushing_epa',
    'receptions',
    'receiving_yards',
    'receiving_yards_after_catch',
    'receiving_tds',
    'receiving_fumbles_lost',
    'receiving_epa',
]
WR_TE_STANDARD_FEATURES = [
    'receptions',
    'receiving_yards',
    'receiving_yards_after_catch',
    'receiving_tds',
    'receiving_fumbles_lost',
    'receiving_epa',
    'wopr',
    'rushing_yards',
    'rushing_tds',
    'rushing_fumbles_lost',
    'rushing_epa',
]
WR_TE_NEXTGEN_FEATURES = [
    'percent_share_of_intended_air_yards',
    'avg_yac',
    'avg_expected_yac',
    'avg_yac_above_expectation',
]
WR_TE_DEFENSE_FEATURES = [
    'receptions',
    'receiving_yards',
    'receiving_air_yards',
    'receiving_yards_after_catch',
    'receiving_tds',
    'receiving_fumbles_lost',
    'receiving_epa',
]
QB_STANDARD_FEATURES = [
    'passing_yards',
    'passing_tds',
    'passing_interceptions',
    'sack_fumbles_lost',
    'passing_epa',
    'rushing_yards',
    'rushing_tds',
    'rushing_fumbles_lost',
    'rushing_epa',
]
QB_NEXTGEN_FEATURES = [
    'avg_air_yards_differential',
]
QB_DEFENSE_FEATURES = [
    'passing_yards',
    'passing_tds',
    'passing_interceptions',
    'sack_fumbles_lost',
    'rushing_yards',
    'rushing_tds',
    'rushing_fumbles_lost',
    'rushing_epa',
]

In [ ]:
FEATURES = {
    'RB': {
        'standard': RB_STANDARD_FEATURES,
        'nextgen': RB_NEXTGEN_FEATURES,
        'defense': RB_DEFENSE_FEATURES,
    },
    'WR': {
        'standard': WR_TE_STANDARD_FEATURES,
        'nextgen': WR_TE_NEXTGEN_FEATURES,
        'defense': WR_TE_DEFENSE_FEATURES,
    },
    'TE': {
        'standard': WR_TE_STANDARD_FEATURES,
        'nextgen': WR_TE_NEXTGEN_FEATURES,
        'defense': WR_TE_DEFENSE_FEATURES,
    },
    'QB': {
        'standard': QB_STANDARD_FEATURES,
        'nextgen': QB_NEXTGEN_FEATURES,
        'defense': QB_DEFENSE_FEATURES,
    }
}

In [ ]:
def add_bye_week_spine(df):

    return pd.MultiIndex.from_product(
        iterables=[
            range(2022, 2026),
            range(1, 15),
            df['team'].unique()
        ],
        names=['season', 'week', 'team']
    ).to_frame(index=False)

In [ ]:
def add_bye_week_table(df):

    spine = add_bye_week_spine(df)
    df = df[['season', 'week', 'team']].drop_duplicates()
    df['joined'] = 'Y'
    joined = pd.merge(
        spine,
        df,
        how='left',
        on=['season', 'week', 'team']
    ).fillna('N')

    return (
        joined[joined['joined'] == 'N']
        .rename({'week': 'bye_week'}, axis=1)
        .drop('joined', axis=1)
    )

In [ ]:
def add_draft_year_table(df):

    return df[['player_id', 'draft_year']].drop_duplicates()

def add_position_spine(df):

    df = df[['season', 'player_id']].drop_duplicates()
    week_df = pd.DataFrame({'week': range(1, 19)})

    return df.merge(week_df, how='cross')

In [ ]:
player_stats = player_stats.sort_values(['season', 'week'])
bye_week_table = add_bye_week_table(team_stats)
draft_year_table = add_draft_year_table(player_info)
team_stats = team_stats.sort_values(['season', 'week'])
position_spine = add_position_spine(player_stats)

In [ ]:
def getitem(d, key):

    return reduce(operator.getitem, key, d)

def get_feature_cols(feat_type, feat_dict):

    features = [
        getitem(feat_dict, (p, feat_type))
        for p in feat_dict
    ]

    return sum(features, [])

In [ ]:
def forward_fill(df, col):

    df[col] = (
        df.groupby('player_id')[col]
        .transform(lambda x: x.ffill())
    )

    return df

def merge_tables(df, keys):

    base_df = df
    for i, (table, key) in enumerate(keys):
        base_df = base_df.merge(table, how='left', on=key)
        if i == 0:
            base_df = forward_fill(base_df, 'team')
            base_df = forward_fill(base_df, 'player')
            base_df = forward_fill(base_df, 'position')

    return base_df

In [ ]:
PLAYER_STATS_COLS = [
    'season',
    'week',
    'player_id',
    'player',
    'position',
    'team',
    'opponent_team',
    'half',
    'full',
    'std'
]
PLAYER_STATS_COLS += list(set(get_feature_cols('standard', FEATURES)))
NEXTGEN_STATS_COLS = ['season', 'week', 'player_id']
NEXTGEN_STATS_COLS += list(set(get_feature_cols('nextgen', FEATURES)))
DEFENSE_STATS_COLS = ['season', 'week', 'team', 'opponent_team']
DEFENSE_STATS_COLS += list(set(get_feature_cols('defense', FEATURES)))
MERGE_KEYS = [
    (player_stats[PLAYER_STATS_COLS], ['season', 'week', 'player_id']),
    (nextgen_stats[NEXTGEN_STATS_COLS], ['season', 'week', 'player_id']),
    (bye_week_table, ['season', 'team']),
    (draft_year_table, 'player_id'),
]

In [ ]:
full_df = merge_tables(position_spine, MERGE_KEYS)
def_df = team_stats[DEFENSE_STATS_COLS]

In [ ]:
GEN_FEATURE_HELPERS = {
    'non_null_games': {
        'val_col': 'half',
        'preprocess': lambda x: x.notnull(),
    },
    'possible_games': {
        'val_col': 'bye_week_flag',
        'preprocess': lambda x: (~x),
    },
}
UNIV_FEATURES = [
    'rolling_durability',
    'returning_from_injury',
    'rookie_flag'
]

In [ ]:
def rolling_feature(df, grp_col, val_col, window, agg_func,
                    preprocess=lambda x: x, min_periods=None,
                    win_type=None, **kwargs):

    return (
        df.groupby(grp_col)[val_col].transform(
            lambda s: getattr(
                preprocess(s)
                .shift(1)
                .rolling(
                    window=window, min_periods=min_periods,
                    win_type=win_type
                ),
                agg_func
            )(**kwargs)
        )
    )

In [ ]:
def simple_transform(df, grp_col, val_col, preprocess):

    return (
        df.groupby(grp_col)[val_col]
        .transform(lambda s: preprocess(s))
    )

In [ ]:
def add_bye_week_flag(df):

    df['bye_week_flag'] = (df['bye_week'] == df['week'])

    return df

def simple_div(df, num, denom):

    return (df[num] / df[denom])

In [ ]:
def rolling_feat_helpers(df, helpers):

    for name in helpers:
        df[name] = rolling_feature(
            df=df,
            grp_col='player_id',
            val_col=helpers[name]['val_col'],
            window=17,
            agg_func='sum',
            min_periods=4,
            preprocess=helpers[name]['preprocess']
        )

    return df

In [ ]:
def add_rookie_flag(df):

    df['draft_year'] = np.where(
        df['draft_year'].notnull(),
        df['draft_year'].astype('Int64'),
        df['draft_year']
    )
    df['rookie_flag'] = df['draft_year'] == df['season']

    return df

In [ ]:
def correct_flags(df):

    df['rookie_flag'] = (
        df['rookie_flag']
        .astype('Int64')
        .fillna(0)
    )
    df['bye_week_flag'] = (
        df['bye_week_flag']
        .astype('Int64')
        .fillna(0)
        .astype('int64')
    )

    return df

In [ ]:
def add_rolling_durability(df):

    df['rolling_durability'] = simple_div(
        df,
        'non_null_games',
        'possible_games'
    )

    return df

In [ ]:
def injury_helper(df, val_col):

    return (
        simple_transform(
            df=df,
            grp_col=['player_id', 'season'],
            val_col=val_col,
            preprocess=lambda x: x.shift(1)
        )
    )

In [ ]:
def add_return_from_injury(df):

    df['pw_bye_week_flag'] = injury_helper(df, 'bye_week_flag')
    df['pw_target'] = injury_helper(df, 'half')
    df['returning_from_injury'] = np.where(
        (df['pw_bye_week_flag'] == 0) &
        (df['pw_target'].isna()) &
        (df['half'].notnull()),
        1,
        0
    )

    return df

In [ ]:
COLS_BEFORE_FUNCS = list(full_df.columns)
full_df = add_bye_week_flag(full_df)
full_df = rolling_feat_helpers(full_df, GEN_FEATURE_HELPERS)
full_df = add_rookie_flag(full_df)
full_df = correct_flags(full_df)
full_df = add_rolling_durability(full_df)
full_df = add_return_from_injury(full_df)
full_df = full_df[
    COLS_BEFORE_FUNCS +
    UNIV_FEATURES +
    ['non_null_games']
]

In [ ]:
class CreateFeatures:

    def __init__(self, full_df, def_df, feats, univ_feats,
                 position='RB', target='half'):

        self._position = position
        self._full_df = full_df[full_df['position'] == position].copy()
        self._def_df = def_df.copy()
        self._keys = ['player_id', 'week', 'season', 'player']
        self._feats = feats[position]
        self._gen_feats = self._feats['standard'] + self._feats['nextgen']
        self._def_feats = self._feats['defense']
        self._univ_feats = univ_feats.copy()
        self._target = target
        self._prepare_data()

    def _rolling_feature(self, df, grp_col, feats, name):

        for f in feats:
            df[name(f)] = rolling_feature(
                df=df,
                grp_col=grp_col,
                val_col=f,
                window=17,
                agg_func='mean',
                min_periods=4,
                win_type='exponential',
                tau=17
            )

        return df

    def _rolling_consistency(self):

        benchmark = 15 if self._position == 'QB' else 10
        self._full_df['games_above_avg'] = rolling_feature(
            df=self._full_df,
            grp_col='player_id',
            val_col=self._target,
            window=17,
            agg_func='sum',
            min_periods=4,
            preprocess=lambda x: (x >= benchmark)
        )
        self._full_df['rolling_consistency'] = simple_div(
            self._full_df,
            'games_above_avg',
            'non_null_games'
        )

    def _marry_datasets(self):

        self._full_df = self._rolling_feature(
            self._full_df,
            'player_id',
            self._gen_feats,
            lambda x: f'rolling_{x}'
        )

        self._def_df = self._rolling_feature(
            self._def_df,
            'opponent_team',
            self._def_feats,
            lambda x: f'def_rolling_{x}'
        )

        self._full_df = self._full_df.merge(
            self._def_df,
            how='left',
            on=['season', 'week', 'opponent_team']
        )

    def _prepare_data(self):

        self._rolling_consistency()
        self._marry_datasets()
        self._gen_feats = [f'rolling_{f}' for f in self._gen_feats]
        self._def_feats = [f'def_rolling_{f}' for f in self._def_feats]
        self._univ_feats += ['rolling_consistency']
        self.data = self._full_df[
            self._keys +
            self._univ_feats +
            self._gen_feats +
            self._def_feats +
            [self._target]
        ]

In [ ]:
pos_datasets = {}
for pos in ['RB', 'QB', 'WR', 'TE']:
    pos_data = CreateFeatures(
        full_df,
        def_df,
        FEATURES,
        UNIV_FEATURES,
        position=pos,
        target='half'
    ).data
    pos_datasets[pos] = pos_data

In [ ]:
def season_week_zip(df):

    season_vals = df['season'].values
    week_vals = df['week'].values

    return list(zip(season_vals, week_vals))

In [ ]:
def rolling_window(df, full_window=51, test_window=17):

    season_week_df = df[['season', 'week']].drop_duplicates()
    season_week_df = season_week_df.sort_values(['season', 'week'])
    season_week_df = season_week_df.iloc[-full_window:]
    window = season_week_zip(season_week_df)

    return [window[:-test_window], window[-test_window:]]

In [ ]:
def training_split(df, windows):

    series = pd.Series(season_week_zip(df))
    df = df.reset_index(drop=True)

    return tuple(df[series.isin(w)] for w in windows)

In [ ]:
class FeatureEngineer:

    def __init__(self, data, target='half'):

        self.data = data
        self._target = target
        self._create_player_names()
        self._ready_data()
        self._corr_matrix = (
            self.data.dropna(subset=target)
            .set_index(['season', 'week', 'player_id'])
            .corr()
        )

    def _create_player_names(self):

        player_df = (
            self.data[['player_id', 'player']]
            .drop_duplicates()
        )
        self.player_names = dict(
            zip(player_df['player_id'], player_df['player'])
        )

    def _ready_data(self):

        windows = rolling_window(self.data)
        self.data = pd.concat(
            list(training_split(self.data, windows)),
            ignore_index=True
        ).drop('player', axis=1)

    def correlations(self, sort_ord='desc'):

        sort_dict = {'asc': True, 'desc': False}

        return (
            self._corr_matrix[self._target]
            .sort_values(ascending=sort_dict[sort_ord])
            .drop(index=self._target)
        )

    def collinearity(self, thres=.5):

        coll_df = self._corr_matrix.drop(columns=self._target)
        coll_df = (
            coll_df.where(
                np.triu(np.ones(coll_df.shape), k=1)
                .astype(bool)
            )
            .stack()
            .sort_values(ascending=False)
            .reset_index()
        )
        coll_df.columns = ['feat_one', 'feat_two', 'collinearity']

        return coll_df[coll_df['collinearity'] > thres]

    def prune_features(self, feats=None):

        if feats:
            self.data = self.data.drop(columns=feats)

    def recent_momentum_feats(self, feats=None):

        if feats:
            for f in feats:
                self.data[f'recent_rolling_{f}'] = rolling_feature(
                    df=self.data,
                    grp_col='player_id',
                    val_col=f,
                    window=3,
                    min_periods=3,
                    agg_func='mean'
                )
                self.data[f'flat_rolling_{f}'] = rolling_feature(
                    df=self.data,
                    grp_col='player_id',
                    val_col=f,
                    window=17,
                    min_periods=4,
                    agg_func='mean'
                )
                self.data[f'momentum_rolling_{f}'] = (
                    self.data[f'recent_rolling_{f}'] -
                    self.data[f'flat_rolling_{f}']
                )
                self.data = self.data.drop(columns=f'flat_rolling_{f}')

In [ ]:
eng_datasets = {}
for pos, df in pos_datasets.items():
    eng_data = FeatureEngineer(df, target='half')
    eng_datasets[pos] = eng_data
    print('-' * len(f'Position: {pos}') * 8)
    print(f'Position: {pos}')
    print('-' * len(f'Position: {pos}') * 8)
    print('\nCorrelation (Ascending)')
    print(eng_data.correlations('asc').head(20).to_markdown())
    print('\nCorrelation (Descending)')
    print(eng_data.correlations('desc').head(20).to_markdown())
    print('\nCollinearity (Descending)')
    print(eng_data.collinearity().head(20).to_markdown() + '\n\n')


------------------------------------------------------------------------------------------------
Position: RB
------------------------------------------------------------------------------------------------

Correlation (Ascending)
|                                         |        half |
|:----------------------------------------|------------:|
| returning_from_injury                   | -0.201182   |
| rookie_flag                             | -0.0567545  |
| def_rolling_receptions                  | -0.00453178 |
| def_rolling_receiving_yards             | -0.0034161  |
| def_rolling_receiving_yards_after_catch | -0.00202288 |
| def_rolling_rushing_fumbles_lost        |  0.00522582 |
| def_rolling_receiving_fumbles_lost      |  0.00534179 |
| rolling_rushing_epa                     |  0.0113403  |
| def_rolling_receiving_epa               |  0.0181213  |
| def_rolling_rushing_epa                 |  0.0207439  |
| def_rolling_rushing_yards               |  0.026644   |
| def_rolling_

In [ ]:
prune_feats = {
    'RB': [
        'rolling_wopr',
        'rolling_receiving_yards',
        'rolling_receiving_yards_after_catch',
        'rolling_rush_yards_over_expected',
        'def_rolling_receiving_yards_after_catch',
        'def_rolling_receiving_yards',
        'def_rolling_receptions',
        'def_rolling_rushing_fumbles_lost',
        'def_rolling_receiving_fumbles_lost',
        'rolling_rushing_epa',
        'def_rolling_receiving_epa',
        'def_rolling_rushing_epa',
        'def_rolling_rushing_yards',
        'def_rolling_receiving_tds',
        'def_rolling_rushing_tds',
    ],
    'QB': [
        "rolling_passing_tds",
        "def_rolling_passing_interceptions",
        "def_rolling_passing_tds",
        "def_rolling_rushing_yards",
        "def_rolling_rushing_fumbles_lost",
        "rolling_avg_air_yards_differential",
        "def_rolling_rushing_tds",
        "rolling_sack_fumbles_lost",
        "rolling_passing_interceptions",
        "def_rolling_passing_yards",
        "def_rolling_rushing_epa",
        "rolling_rushing_fumbles_lost",
        "def_rolling_sack_fumbles_lost",
    ],
    'WR': [
        "rolling_wopr",
        "rolling_receptions",
        "rolling_receiving_yards_after_catch",
        "def_rolling_receiving_tds",
        "rolling_rushing_fumbles_lost",
        "def_rolling_receiving_fumbles_lost",
        "rolling_rushing_epa",
        "def_rolling_receiving_yards_after_catch",
        "def_rolling_receiving_epa",
        "def_rolling_receptions",
        "def_rolling_receiving_yards",
        "def_rolling_receiving_air_yards",
        "rolling_rushing_yards",
    ],
    'TE': [
        "rolling_receiving_yards",
        "rolling_receptions",
        "rolling_rushing_epa",
        "def_rolling_receptions",
        "def_rolling_receiving_air_yards",
        "rolling_avg_expected_yac",
        "def_rolling_receiving_tds",
        "def_rolling_receiving_yards",
        "def_rolling_receiving_yards_after_catch",
        "def_rolling_receiving_epa",
        "rolling_rushing_fumbles_lost",
        "rolling_rushing_tds",
        "rolling_rushing_yards",
        "rolling_receiving_fumbles_lost",
    ],
}

In [ ]:
momentum_feats = {
    'RB': [
        'rolling_consistency',
		'rolling_receptions',
    ],
    'QB': [
        'rolling_consistency',
		'rolling_durability',
    ],
    'WR': [
        'rolling_receiving_yards',
        'rolling_consistency',
    ],
    'TE': [
        'rolling_wopr',
    ],
}

In [ ]:
for pos, feats in prune_feats.items():
    eng_datasets[pos].prune_features(feats)
    eng_datasets[pos].recent_momentum_feats(momentum_feats[pos])

In [ ]:
rdy_datasets = {}
for pos in ['RB', 'QB', 'WR', 'TE']:
    rdy_datasets[pos] = eng_datasets[pos].data

## Model Training

In [ ]:
drive.mount('/content/drive', force_remount=True)
FILE_PATH = '/content/drive/MyDrive/Football/'
FILE_NAME = 'Fantasy Football Predictions.xlsx'

Mounted at /content/drive


In [ ]:
class ModelPipeline:

    def __init__(self, df, file_path, position='RB', target='half',
                 train=True):

        self.position = position
        self._train = train
        self._df = df.dropna(subset=target)
        self._file_path = file_path
        self._target = target
        self._model = make_pipeline(
            SimpleImputer(strategy='mean'),
            RandomForestRegressor(
                random_state=42,
                criterion='absolute_error'
            )
        )
        self._param_dist = {
            'randomforestregressor__max_depth': randint(3, 10),
            'randomforestregressor__n_estimators': randint(100, 500),
            'randomforestregressor__min_samples_leaf': randint(2, 10),
            'randomforestregressor__min_samples_split': randint(2, 10),
        }
        self._split_data()
        if self._train:
            self._begin_search()
            self._score()
            self._fit_best_model()
        else:
            self._load_model()

    def _ready_X(self, df):

        return (
            df.drop(columns=self._target)
            .set_index(['season', 'week', 'player_id'])
        )

    def _ready_y(self, df):

        return (
            df[['season', 'week', 'player_id', self._target]]
            .set_index(['season', 'week', 'player_id'])
        )

    def _split_data(self):

        windows = rolling_window(self._df)
        train, test = training_split(self._df, windows)
        self._X_train, self._X_test = (
            self._ready_X(train),
            self._ready_X(test),
        )
        self._y_train, self._y_test = (
            self._ready_y(train),
            self._ready_y(test),
        )

    def _ravel(self, arr):

        return arr.values.ravel()

    def _begin_search(self):

        start = time.time()
        print('-' * len(f'{self.position} search started.') * 2)
        print(f'{self.position} search started.')
        self._search = RandomizedSearchCV(
            self._model,
            self._param_dist,
            random_state=42,
            scoring='neg_mean_absolute_error',
            n_iter=10,
            cv=3,
            n_jobs=-1
        ).fit(
            self._X_train,
            self._ravel(self._y_train)
        )
        end = time.time()
        time_elapsed = (end - start) / 60
        complete = f'{self.position} search complete.\n'
        elapsed = f'Time elapsed: {time_elapsed: 2f}'
        print(complete + elapsed)
        self._best_estimator = self._search.best_estimator_

    def _score(self):

        y_pred = self._best_estimator.predict(self._X_test)
        self.mae = mean_absolute_error(
            self._ravel(self._y_test),
            y_pred
        )
        self.r2 = r2_score(
            self._ravel(self._y_test),
            y_pred
        )
        print(f'MAE score: {self.mae:.2f}\n' + f'R2 score: {self.r2:.2f}')

    def _fit_best_model(self):

        X = pd.concat([self._X_train, self._X_test])
        y = pd.concat([self._y_train, self._y_test])
        self.best_model = self._best_estimator.fit(X, self._ravel(y))
        dump(
            self.best_model,
            f'{self._file_path}{self.position}_model.pkl'
        )
        print(f'{self.position} model saved.')

    def _load_model(self):

        self.best_model = load(
            f'{self._file_path}{self.position}_model.pkl'
        )
        print('-' * len(f'{self.position} model loaded.') * 2)
        print(f'{self.position} model loaded.')
        self._best_estimator = self.best_model
        self._score()

In [ ]:
pos_pipeline = {}
for pos, df in rdy_datasets.items():
    pos_model = ModelPipeline(
        df=df,
        position=pos,
        file_path=FILE_PATH,
        target='half',
        train=False
    ).best_model
    pos_pipeline[pos] = {'model': pos_model, 'df': df}

--------------------------------
RB model loaded.
MAE score: 3.91
R2 score: 0.39
--------------------------------
QB model loaded.
MAE score: 5.62
R2 score: 0.40
--------------------------------
WR model loaded.
MAE score: 3.43
R2 score: 0.33
--------------------------------
TE model loaded.
MAE score: 2.91
R2 score: 0.26


In [ ]:
class Inference:

    def __init__(self, model, df, player_names, file_path, file_name,
                 position='RB', target='half'):

        self._model = model
        self._df = df
        self._player_names = player_names
        self._full_path = file_path + file_name
        self._position = position
        self._target = target
        self._ready_df()
        self._predict()
        self._create_final_df()
        self._merge_predictions()

    def _ready_df(self):

        curr_df = self._df.sort_values(['season', 'week'])
        seas = curr_df['season'].values[-1]
        wk = curr_df['week'].values[-1]
        self._df = self._df.query('season == @seas and week == @wk')

    def _predict(self):

        X = (
            self._df.set_index(['season', 'week', 'player_id'])
            .drop(columns=self._target)
        )
        self._predictions = self._model.predict(X)

    def _create_final_df(self):

        next_seas = (
            self._df['season'].max() + 1
            if self._df['week'].max() == 18
            else self._df['season'].max()
        )
        next_wk = (
            1 if self._df['week'].max() == 18
            else self._df['week'].max() + 1
        )
        self.week = f'{next_seas}-{str(next_wk).zfill(2)}'
        self.final_df = self._df[['player_id']].copy()
        self.final_df['Position'] = self._position
        self.final_df['Player'] = (
            self.final_df['player_id']
            .map(self._player_names)
        )
        self.final_df[self.week] = self._predictions.round(2)
        self.final_df = (
            self.final_df.drop(columns='player_id')
            .sort_values(self.week, ascending=False)
        )

    def _merge_predictions(self):

        if os.path.isfile(self._full_path):
            existing_df = pd.read_excel(
                self._full_path,
                sheet_name='Predictions'
            )
            self.final_df = self.final_df.merge(
                existing_df,
                how='outer',
                on=['Position', 'Player']
            )
        if self._target == 'std':
            target = f' {self._target} scoring predictions created.'
        else:
            target = f' {self._target}-ppr scoring predictions created.'
        print(self._position + target)

In [ ]:
pos_player_names = {}
for pos, obj in eng_datasets.items():
    pos_player_names[pos] = obj.player_names

In [ ]:
final_dfs = []
curr_season, curr_week = set(), set()
for pos in pos_pipeline:
    model = pos_pipeline[pos]['model']
    df = pos_pipeline[pos]['df']
    player_names = pos_player_names[pos]
    final_df = Inference(
        model=model,
        df=df,
        player_names=player_names,
        file_path=FILE_PATH,
        file_name=FILE_NAME,
        position=pos
    )
    season, week = final_df.week.split('-')
    curr_season.add(season)
    curr_week.add(week)
    final_dfs.append(final_df.final_df)

RB half-ppr scoring predictions created.
QB half-ppr scoring predictions created.
WR half-ppr scoring predictions created.
TE half-ppr scoring predictions created.


In [ ]:
pd.concat(final_dfs, ignore_index=True).to_excel(
    FILE_PATH + FILE_NAME,
    sheet_name='Predictions',
    index=False
)
print(
    f'Season: {list(curr_season)[0]}\n' +
    f'Week: {list(curr_week)[0]}\n' +
    'Predictions saved.'
)

Season: 2026
Week: 01
Predictions saved.
